# Семинар: итераторы и генераторы

# Задача 1

На прошлом семинаре вы с Матвеем парсили CSV-файл с ценами и данными разных активов. Вот как это было:

In [25]:
from typing import Any
import csv


def read_portfolio(filename):
    '''
    Read a stock portfolio file into a list of dictionaries with keys
    name, shares, and price.
    '''
    portfolio = []
    with open(filename) as f:
        rows = csv.reader(f)
        headers = next(rows)

        for row in rows:
            record = dict(zip(headers, row))
            stock = {
                 'name'   : record['name'],
                 'shares' : int(record['shares']),
                 'price'   : float(record['price'])
            }
            portfolio.append(stock)

    return portfolio


def read_prices(filename):
    '''
    Read a CSV file of price data into a dict mapping names to prices.
    '''
    prices = {}
    with open(filename) as f:
        rows = csv.reader(f)
        for row in rows:
            try:
                prices[row[0]] = float(row[1])
            except IndexError:
                pass

    return prices


def make_report_data(portfolio, prices) -> list[tuple[str, int, float, float]]:
    '''
    Make a list of (name, shares, price, change) tuples given a portfolio list
    and prices dictionary.
    '''
    rows = []
    for stock in portfolio:
        current_price = prices[stock['name']]
        change        = current_price - stock['price']
        summary       = (stock['name'], stock['shares'], current_price, change)
        rows.append(summary)
    return rows


def print_report(report: list[tuple[str, int, float, float]]) -> None:
    headers = ('Name', 'Shares', 'Price', 'Change')
    print('%10s %10s %10s %10s' % headers)
    print(('-' * 10 + ' ') * len(headers))
    for row in report:
        print('%10s %10d %10.2f %10.2f' % row)


def portfolio_report(portfolio_filename: str, prices_filename: str) -> None:
    portfolio = read_portfolio(portfolio_filename)
    prices    = read_prices(prices_filename)
    report    = make_report_data(portfolio, prices)
    print_report(report)


portfolio_report(
    '../part7_program_organization/portfolio.csv',
    '../part7_program_organization/prices.csv',
)

      Name     Shares      Price     Change
---------- ---------- ---------- ---------- 
        AA        100       9.22     -22.98
       IBM         50     106.28      15.18
       CAT        150      35.46     -47.98
      MSFT        200      20.89     -30.34
        GE         95      13.48     -26.89
      MSFT         50      20.89     -44.21
       IBM        100     106.28      35.84


Долой обычные словари! Даёшь объектную модель!

In [ ]:
class Stock:
    def __init__(self, name, shares, price):
        self.name = name
        self.shares = shares
        self.price = price


class Portfolio:
    def __init__(self, holdings: list[Stock]):
        self._holdings = holdings

    @property
    def total_cost(self):
        return sum([s.cost for s in self._holdings])

    def tabulate_shares(self):
        from collections import Counter
        total_shares = Counter()
        for s in self._holdings:
            total_shares[s.name] += s.shares
        return total_shares

In [27]:
# fileparse.py
import csv

def parse_csv(lines, select=None, types=None, has_headers=True, delimiter=',', silence_errors=False):
    '''
    Parse a CSV file into a list of records with type conversion.
    '''
    if select and not has_headers:
        raise RuntimeError('select requires column headers')

    rows = csv.reader(lines, delimiter=delimiter)

    # Read the file headers (if any)
    headers = next(rows) if has_headers else []

    # If specific columns have been selected, make indices for filtering and set output columns
    if select:
        indices = [ headers.index(colname) for colname in select ]
        headers = select

    records = []
    for rowno, row in enumerate(rows, 1):
        if not row:     # Skip rows with no data
            continue

        # If specific column indices are selected, pick them out
        if select:
            row = [ row[index] for index in indices]

        # Apply type conversion to the row
        if types:
            try:
                row = [func(val) for func, val in zip(types, row)]
            except ValueError as e:
                if not silence_errors:
                    print(f"Row {rowno}: Couldn't convert {row}")
                    print(f"Row {rowno}: Reason {e}")
                continue

        # Make a dictionary or a tuple
        if headers:
            record = dict(zip(headers, row))
        else:
            record = tuple(row)
        records.append(record)

    return records

In [28]:
def read_portfolio(filename) -> Portfolio:
    '''
    Read a stock portfolio file into a list of dictionaries with keys
    name, shares, and price.
    '''
    with open(filename) as file:
        portdicts = parse_csv(file,
                                        select=['name','shares','price'],
                                        types=[str,int,float])

    portfolio = [Stock(d['name'], d['shares'], d['price']) for d in portdicts]
    return Portfolio(portfolio)


def portfolio_report(portfolio_filename: str, prices_filename: str) -> None:
    portfolio = read_portfolio(portfolio_filename)
    prices    = read_prices(prices_filename)
    report    = make_report_data(portfolio, prices)
    print_report(report)

In [29]:
portfolio_report(
    '../part7_program_organization/portfolio.csv',
    '../part7_program_organization/prices.csv',
)

TypeError: 'Portfolio' object is not iterable

# Задача 1.2

А давайте ещё чуть расширим наш класс Portfolio.

```python
portfolio = read_portfolio("../part7_program_organization/portfolio.csv")
print(len(portfolio)) - количество записей Stock
print(portfolio[1]) - обращение к Stock по индексу
print("IBM" in portfolio) - проверка, что такой Stock встречается
```

In [ ]:
# Какие методы и как нужно добавить?

class Portfolio:
    def __init__(self, holdings):
        self._holdings = holdings

    @property
    def total_cost(self):
        return sum([s.cost for s in self._holdings])

    def tabulate_shares(self):
        from collections import Counter
        total_shares = Counter()
        for s in self._holdings:
            total_shares[s.name] += s.shares
        return total_shares

In [37]:
portfolio = Portfolio([
    Stock("SBER", 100, 15),
    Stock("TATN", 10, 100),
])

print("MTSS" in portfolio)

False


# Задача 2

Давайте сделаем свой reversed-итератор для Iterable:

In [ ]:
reversed([])

In [ ]:
from collections.abc import Reversible


3
2
1


# Задача 3

Знаком ли вам zip?

In [18]:
a = (1, 2)
b = (3, 4, 5)
print(list(zip(a, b)))

[(1, 3), (2, 4)]


Наша задача сделать класс MyZip с аналогичным поведением, но своими руками (без мам, пап и кредитов)

In [ ]:
class MyZip: ...

# Задача 4:

Необходимо создать генератор read_file_in_chunks, который считывает файл по несколько строк.

Генератор ожидает как аргумент имя файла и количество строк, которые нужно считать за раз
и должен возвращать указанное количество строк одной строкой на каждой итерации.
Строки файла должны считываться по необходимости, то есть нельзя считывать все строки файла за раз.

Параметры функции:

* filename - имя файла из которого считываются строки
* chunk_size - сколько строк считывать за раз

Проверить работу генератора на примере файла text.txt

Убедиться, что все строки возвращаются. Если в последней итерации строк меньше, чем указано в chunk_size,
они всё равно должны возвращаться. После того как строки закончились, генератор должен останавливаться.

Ограничение: нельзя использовать функции из модуля itertools.

In [ ]:
from collections.abc import Generator



In [21]:
chunk_reader = read_file_in_chunks("_resources/text.txt", 2)

In [22]:
first_chunk = next(chunk_reader)
second_chunk = next(chunk_reader)
print(first_chunk)
print(second_chunk)

['Я волком бы\n', 'выгрыз\n']
['бюрократизм.\n', 'К мандатам\n']


---


In [12]:
first = [i for i in range(10000)]
second = (i for i in range(10000))

In [55]:
import timeit
from functools import reduce, partial
from operator import add

N = 10_000_000

# Генераторное выражение внутри sum()
gen_sum = timeit.timeit(lambda: reduce(add, (x for x in range(N))), number=10)
print(f"Генератор + sum: {gen_sum:.4f} сек")

# # Списковое включение сначала создаёт список, потом sum()
list_sum = timeit.timeit(lambda: reduce(add, [x for x in range(N)]), number=10)
print(f"Список + sum: {list_sum:.4f} сек")

Генератор + sum: 3.6736 сек
Список + sum: 3.7340 сек


In [21]:
import tracemalloc

N = 10_000_000

# --- Списковое включение ---
tracemalloc.start()
list_comp = [x for x in range(N)]
current, peak = tracemalloc.get_traced_memory()
print(f"Список: текущая память = {current / 1024**2:.2f} МБ, пик = {peak / 1024**2:.2f} МБ")
tracemalloc.stop()

# --- Генераторное выражение ---
tracemalloc.start()
gen_exp = (x for x in range(N))
# Память под сам генератор (несколько десятков байт)
current, peak = tracemalloc.get_traced_memory()
print(f"Генератор (до обхода): текущая память = {current / 1024**2:.2f} МБ, пик = {peak / 1024**2:.2f} МБ")

# Теперь обойдём генератор (память не растёт)
for _ in gen_exp:
    pass
current, peak = tracemalloc.get_traced_memory()
print(f"Генератор (после обхода): текущая память = {current / 1024**2:.2f} МБ, пик = {peak / 1024**2:.2f} МБ")
tracemalloc.stop()

Список: текущая память = 390.14 МБ, пик = 390.16 МБ
Генератор (до обхода): текущая память = 0.00 МБ, пик = 0.03 МБ
Генератор (после обхода): текущая память = 0.00 МБ, пик = 0.03 МБ
